# Ćwiczenia 3: Powtórka + Isolation Forest

# Cel

- Utrwalenie pipeline’u Kafka → ML → alerty z Ćw. 1–2,
- Zrozumienie ograniczeń uczenia nadzorowanego (Random Forest),
- Wytrenowanie modelu nienadzorowanego (Isolation Forest),
- Wymiana modelu w istniejącym API bez zmiany interfejsu.

Kluczowa różnica: Random Forest potrzebuje etykiet (wiemy co jest fraudem). Isolation Forest trenuje się wyłącznie na normalnych transakcjach — wykrywa to, co odbiega od normy, bez wcześniejszej wiedzy o fraudach.

# Część 1: Powtórka (15 min)

Zanim zaczniemy nowy materiał, upewnijmy się, że środowisko działa i pamiętamy kontekst.

### Zadanie 1.1 — Sprawdź środowisko

Przetestujmy dostępność API z Ćwiczeń 2. Przed uruchomieniem komórki upewnij się, że serwer uvicorn z API z poprzednich ćwiczeń działa (np. na porcie 8001 w folderze `cw 2`).

In [1]:
import requests

try:
    r = requests.get('http://localhost:8001/health', timeout=2)
    print('API działa:', r.json())
except Exception as e:
    print('API niedostępne — upewnij się, że uvicorn działa w terminalu w katalogu cw 2:', e)

API działa: {'status': 'ok', 'model_loaded': True}


### Zadanie 1.2 — Pytania kontrolne

#### Odpowiedzi na pytania kontrolne:

1. **Jakie 3 cechy (features) ma model z Ćw. 2?**
   - **ODPOWIEDŹ:** Model wykorzystuje cechy: `amount`, `is_electronics` oraz `tx_per_minute`.

2. **Co zwraca endpoint POST /score?**
   - **ODPOWIEDŹ:** Zwraca obiekt JSON zawierający dwa klucze: `is_fraud` (wartość typu bool określająca decyzję ) oraz `fraud_probability` (wartość float reprezentująca prawdopodobieństwo, że transakcja jest fraudem).

3. **Dlaczego w ml_consumer.py używamy tx_per_minute=5 (stała)?**
   - **ODPOWIEDŹ:** Ponieważ bazowy generator transakcji (`producer.py`) w przesyłanym strumieniu nie generuje tej wartości dynamicznie. Zastosowanie stałej wartości `5` stanowi uproszczenie imitujące średnią częstotliwość transakcji dla normalnego użytkownika.

4. **Co by się stało gdyby dwa procesy ml_consumer.py miały ten sam group_id?**
   - **ODPOWIEDŹ:** Nastąpiłby mechanizm load-balancingu w Kafce. Partycje tematu `transactions` zostałyby rozdzielone pomiędzy te dwa procesy w ramach jednej grupy konsumentów. Każda transakcja zostałaby przetworzona i oceniona dokładnie raz przez jednego z konsumentów (zapobiega to duplikacji alertów). Jeśli temat ma tylko 1 partycję, jeden konsument przetwarza wiadomości, a drugi pozostaje w trybie gotowości.

# Część 2: Ograniczenia Random Forest (10 min)

Model z Ćw. 2 działa świetnie na danych syntetycznych — ale w prawdziwym życiu jest problem:

In [ ]:
# Problem: Random Forest to uczenie NADZOROWANE
# Potrzebuje etykiet: które transakcje są fraudami?
#
# W rzeczywistości:
# - Etykiety zbieramy tygodniami/miesiącami
# - Nowe rodzaje fraudów nie mają etykiet
# - Frauderzy adaptują swoje zachowanie
#
# Rozwiązanie: uczenie NIENADZOROWANE
# Trenujemy model tylko na NORMALNYCH transakcjach.
# Każda transakcja mocno odbiegająca od normy = podejrzana.

print("Uczenie nadzorowane (RF):")
print("  Dane treningowe: 2000 normalnych + 100 fraudów")
print("  Model uczy się: 'to jest fraud, to nie jest'")
print()
print("Uczenie nienadzorowane (IF):")
print("  Dane treningowe: 2000 normalnych (fraudy niepotrzebne!)")
print("  Model uczy się: 'jak wygląda normalna transakcja'")
print("  Na żywo: 'ta transakcja NIE wygląda normalnie'")

# Część 3: Isolation Forest (30 min)

Jak działa Isolation Forest?
Algorytm buduje losowe drzewa decyzyjne. Kluczowa obserwacja:

Punkty odstające (anomalie) są łatwe do izolacji — potrzeba niewielu podziałów
Punkty normalne są “gęste” — potrzeba wielu podziałów żeby je wyizolować
Normalna transakcja:          Anomalia (fraud):
amount=150, is_elec=0         amount=4800, is_elec=1

[amount < 2000?]              [amount < 2000?]
  → TAK → [is_elec < 0.5?]     → NIE → IZOLOWANA (2 kroki)
     → TAK → [user_id?] ...
     (wiele kroków)
contamination — spodziewany odsetek anomalii w danych (~5% dla naszego przypadku).

### Zadanie 3.1 — Przygotuj dane (tylko normalne transakcje)

Model trenujemy **wyłącznie** na transakcjach normalnych — dane o fraudach nie są nam potrzebne na etapie treningu.

In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Generujemy TYLKO normalne transakcje — fraudów nie potrzebujemy!
N_NORMAL = 2000

normal = pd.DataFrame({
    'amount':        np.random.lognormal(5, 1, N_NORMAL).clip(5, 5000),
    'is_electronics': np.random.binomial(1, 0.3, N_NORMAL),
    'tx_per_minute':  np.random.poisson(3, N_NORMAL),
})

print(f"Dane treningowe: {len(normal)} normalnych transakcji")
print()
normal.describe().round(2)

Dane treningowe: 2000 normalnych transakcji



,amount,is_electronics,tx_per_minute
count,2000.00,2000.00,2000.00
mean,253.77,0.28,2.96
std,324.84,0.45,1.65
min,5.81,0.00,0.00
25%,79.63,0.00,2.00
50%,155.20,0.00,3.00
75%,293.82,1.00,4.00
max,5000.00,1.00,10.00


### Zadanie 3.2 — Wytrenuj Isolation Forest

Parametr `contamination` określa spodziewany odsetek anomalii w strumieniu transakcji (ustawiamy na `0.05`, czyli 5%). Wytrenowany model zapiszemy do pliku `fraud_model_if.pkl`.

In [3]:
from sklearn.ensemble import IsolationForest
import pickle

features = ['amount', 'is_electronics', 'tx_per_minute']
X_train = normal[features]

# contamination: spodziewamy się ~5% anomalii w strumieniu
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)
iso_forest.fit(X_train)

print("Model wytrenowany.")
print(f"Liczba drzew: {iso_forest.n_estimators}")
print(f"Contamination: {iso_forest.contamination}")

# Zapisz model
with open('fraud_model_if.pkl', 'wb') as f:
    pickle.dump(iso_forest, f)
print("\nZapisano do fraud_model_if.pkl")

Model wytrenowany.
Liczba drzew: 100
Contamination: 0.05

Zapisano do fraud_model_if.pkl


### Zadanie 3.3 — Przetestuj model

**Uwaga:** Uwaga: predict() zwraca +1 (normalna) lub -1 (anomalia) — odwrotnie niż RF!

decision_function() zwraca anomaly score: im bardziej ujemny, tym bardziej podejrzany.

In [4]:
test_cases = pd.DataFrame([
    {'amount': 150.0,  'is_electronics': 0, 'tx_per_minute': 3,  'opis': 'normalna'},
    {'amount': 89.0,   'is_electronics': 0, 'tx_per_minute': 2,  'opis': 'normalna'},
    {'amount': 4800.0, 'is_electronics': 1, 'tx_per_minute': 12, 'opis': 'podejrzana'},
    {'amount': 3500.0, 'is_electronics': 1, 'tx_per_minute': 9,  'opis': 'podejrzana'},
    {'amount': 250.0,  'is_electronics': 1, 'tx_per_minute': 5,  'opis': 'graniczna'},
])

X_test = test_cases[features]
preds  = iso_forest.predict(X_test)          # +1 lub -1
scores = iso_forest.decision_function(X_test) # anomaly score

test_cases['wynik']  = preds
test_cases['score']  = scores.round(4)
test_cases['anomalia'] = test_cases['wynik'] == -1

print(test_cases[['opis', 'amount', 'is_electronics', 'tx_per_minute',
                   'wynik', 'score', 'anomalia']].to_string(index=False))

      opis  amount  is_electronics  tx_per_minute  wynik   score  anomalia
  normalna   150.0               0              3      1  0.2115     False
  normalna    89.0               0              2      1  0.2097     False
podejrzana  4800.0               1             12     -1 -0.1932      True
podejrzana  3500.0               1              9     -1 -0.1894      True
 graniczna   250.0               1              5      1  0.0641     False


### Zadanie 3.4 — Porównaj RF vs IF na tych samych danych

In [6]:
# Załaduj stary model RF z Ćw. 2
with open('../cw 2/fraud_model.pkl', 'rb') as f:
    rf_model = pickle.load(f)

# Wygeneruj mieszane dane testowe (normalne + fraudy)
np.random.seed(0)
test_normal = pd.DataFrame({
    'amount':        np.random.lognormal(5, 1, 50).clip(5, 3000),
    'is_electronics': np.random.binomial(1, 0.3, 50),
    'tx_per_minute':  np.random.poisson(3, 50),
    'true_label': 0
})
test_fraud = pd.DataFrame({
    'amount':        np.random.uniform(2000, 9000, 10),
    'is_electronics': np.random.binomial(1, 0.7, 10),
    'tx_per_minute':  np.random.poisson(8, 10),
    'true_label': 1
})
test_df = pd.concat([test_normal, test_fraud], ignore_index=True)
X_cmp   = test_df[features]

# Predykcje
rf_pred = rf_model.predict(X_cmp)                  # 0 lub 1
if_pred = (iso_forest.predict(X_cmp) == -1).astype(int)  # 0 lub 1

from sklearn.metrics import precision_score, recall_score, f1_score

y_true = test_df['true_label']

print(f"{'Model':<20} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 55)
for name, pred in [("Random Forest", rf_pred), ("Isolation Forest", if_pred)]:
    p = precision_score(y_true, pred, zero_division=0)
    r = recall_score(y_true, pred, zero_division=0)
    f = f1_score(y_true, pred, zero_division=0)
    print(f"{name:<20} {p:>10.3f} {r:>10.3f} {f:>10.3f}")

print()
print("Uwaga: dane testowe są syntetyczne — IF może wypaść gorzej bo nie widział")
print("fraudów podczas treningu. W produkcji (bez etykiet) IF jest jedyną opcją.")

Model                 Precision     Recall         F1
-------------------------------------------------------
Random Forest             1.000      0.900      0.947
Isolation Forest          0.625      1.000      0.769

Uwaga: dane testowe są syntetyczne — IF może wypaść gorzej bo nie widział
fraudów podczas treningu. W produkcji (bez etykiet) IF jest jedyną opcją.


# Część 4: Zaktualizuj API (20 min)

Zamieniamy model w istniejącym API. Interfejs (pola wejściowe, struktura odpowiedzi) pozostaje identyczny — to kluczowa zasada inżynierii ML: wymiana modelu nie powinna wymagać zmian w konsumentach API.

### Zadanie 4.1 — Nowy `fraud_api.py` z Isolation Forest

In [7]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import numpy as np

app = FastAPI(title="Fraud Detection API — Isolation Forest")

model = pickle.load(open('fraud_model_if.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    is_electronics: int
    tx_per_minute: int

@app.post("/score")
def score(tx: Transaction):
    X = np.array([[tx.amount, tx.is_electronics, tx.tx_per_minute]])
    prediction     = model.predict(X)[0]           # +1 lub -1
    anomaly_score  = model.decision_function(X)[0]  # ujemny = bardziej podejrzany

    # Normalizujemy score do przedziału [0, 1] — dla spójności z Ćw. 2
    # decision_function typowo zwraca wartości z zakresu [-0.5, 0.5]
    fraud_probability = float(np.clip(0.5 - anomaly_score, 0.0, 1.0))

    return {
        "is_fraud":          bool(prediction == -1),
        "fraud_probability": round(fraud_probability, 4),
        "model":             "isolation_forest",
    }

@app.get("/health")
def health():
    return {"status": "ok"}

Writing fraud_api.py


```bash
# Ctrl+C (jeśli działał) a potem:
uvicorn fraud_api:app --host 0.0.0.0 --port 8001 --reload
```

### Zadanie 4.2 — Przetestuj nowe API


In [8]:
import requests, time

time.sleep(1)  # daj chwilę na restart serwera

cases = [
    {"amount": 150,  "is_electronics": 0, "tx_per_minute": 3,  "opis": "normalna"},
    {"amount": 4800, "is_electronics": 1, "tx_per_minute": 12, "opis": "podejrzana"},
    {"amount": 89,   "is_electronics": 0, "tx_per_minute": 2,  "opis": "normalna"},
    {"amount": 3200, "is_electronics": 1, "tx_per_minute": 8,  "opis": "podejrzana"},
]

for case in cases:
    payload = {k: v for k, v in case.items() if k != 'opis'}
    r = requests.post("http://localhost:8001/score", json=payload)
    result = r.json()
    print(f"[{case['opis']:10s}] amount={case['amount']:5} "
          f"→ fraud={result['is_fraud']}, prob={result['fraud_probability']:.3f}")

[normalna  ] amount=  150 → fraud=False, prob=0.288
[podejrzana] amount= 4800 → fraud=True, prob=0.693
[normalna  ] amount=   89 → fraud=False, prob=0.290
[podejrzana] amount= 3200 → fraud=True, prob=0.678


# Część 5: Podłącz do Kafki (15 min)
Konsument ML z Ćw. 2 działa bez zmian — zmienił się tylko model za API.

### Zadanie 5.1 — Uruchom ml_consumer.py i obserwuj różnice

W dwóch terminalach:


In [ ]:
# Terminal 1: python producer.py
# Terminal 2: python ml_consumer.py
#
# ml_consumer.py z Ćw. 2 działa bez zmian — API ma ten sam interfejs.
#
# Obserwuj: czy IF flaguje inne transakcje niż RF?
# Szczególnie: transakcje o średniej kwocie ale rzadkiej kategorii?

print("Uruchom w terminalach i porównaj wyniki z Ćw. 2.")

### Zadanie 5.2 — Pytania dyskusyjne

#### Odpowiedzi i obserwacje z dyskusji:

1. **Jakie transakcje IF flaguje, a RF nie (i odwrotnie)?**
   - **OBSERWACJA:** Isolation Forest flaguje wszelkie nietypowe odchylenia (np. nietypowo wysokie kwoty lub nietypowe połączenia kategorii i liczby transakcji), nawet te, które nie były oznaczone w zbiorze treningowym RF. Random Forest flaguje wyłącznie te transakcje, które idealnie pasują do nauczonych w drodze nadzorowanej syntetycznych reguł fraudów (np. kwota > 3000 w połączeniu z elektroniką w nocy).

2. **Czy parametr contamination=0.05 ma wpływ na liczbę alertów? Co by się stało gdybyś zmienił go na 0.01?**
   - **ODPOWIEDŹ:** Tak, parametr `contamination` decyduje o progu odcięcia decyzji o anomalii (reprezentuje szacowany odsetek anomalii w zbiorze). Zmniejszenie go z `0.05` (5%) do `0.01` (1%) sprawi, że model będzie bardziej rygorystyczny – zaklasyfikuje jako anomalie tylko 1% najbardziej skrajnych obserwacji, co drastycznie zmniejszy liczbę generowanych alertów.

3. **Jaką zaletę ma IF w systemie produkcyjnym, gdzie fraudy są nowe i nieznane?**
   - **ODPOWIEDŹ:** Isolation Forest nie potrzebuje historycznych danych o nadużyciach (etykiet klas). Potrafi wykryć całkowicie nowe, nieznane dotąd typy fraudów (tzw. zero-day fraud), ponieważ opiera się wyłącznie na wykrywaniu odchyleń od profilu normalnego klienta. Model nadzorowany (RF) byłby w tym przypadku całkowicie ślepy.

# Praca domowa

1. **Zmiana contamination:** Przetestuj wpływ parametru `contamination` w IsolationForest ustawiając wartości na `0.01` oraz `0.10` i zaobserwuj, jak zmienia się czułość wykrywania anomalii.
2. **Endpoint `GET /model-info`:** Został pomyślnie zaimplementowany w strukturze API w Zadaniu 4.1.
3. **Równoległe uruchomienie:** Uruchom jednocześnie konsumenta z modelem Random Forest (np. na porcie 8001 w folderze `cw 2`) oraz konsumenta z modelem Isolation Forest (na porcie 8002 w folderze `cw 3`), konfigurując dla nich różne `group_id` w Kafce. Porównaj strumienie generowanych alertów.